In [0]:
from pyspark.sql.functions import *

# ==========================================================
# Read Silver Tables
# ==========================================================

inpatient = spark.read.table(
    "healthcare_claims_catalog.silver.inpatient"
)

dim_beneficiary = spark.read.table(
    "healthcare_claims_catalog.gold.dim_beneficiary"
)

dim_provider = spark.read.table(
    "healthcare_claims_catalog.gold.dim_provider"
)

dim_date = spark.read.table(
    "healthcare_claims_catalog.gold.dim_date"
)

dim_diagnosis = spark.read.table(
    "healthcare_claims_catalog.gold.dim_diagnosis"
)

# ==========================================================
# Create Fact
# (dim_beneficiary join is safe now — one row per DESYNPUF_ID after
# the dim_beneficiary fix)
# ==========================================================

fact_inpatient = (
    inpatient

    .join(
        dim_beneficiary.select(
            "DESYNPUF_ID",
            "BENEFICIARY_KEY"
        ),
        "DESYNPUF_ID",
        "left"
    )

    .join(
        dim_provider.select(
            "PROVIDER_ID",
            "PROVIDER_KEY"
        ),
        inpatient.PRVDR_NUM == dim_provider.PROVIDER_ID,
        "left"
    )

    .join(
        dim_diagnosis.select(
            "DIAGNOSIS_CODE",
            "DIAGNOSIS_KEY"
        ),
        inpatient.ICD9_DGNS_CD_1 == dim_diagnosis.DIAGNOSIS_CODE,
        "left"
    )
)

# ==========================================================
# Date Keys
# ==========================================================

fact_inpatient = (

    fact_inpatient

    .withColumn(
        "ADMISSION_DATE_KEY",
        date_format("CLM_FROM_DT","yyyyMMdd").cast("int")
    )

    .withColumn(
        "DISCHARGE_DATE_KEY",
        date_format("CLM_THRU_DT","yyyyMMdd").cast("int")
    )
)

# ==========================================================
# Length of Stay
# ==========================================================

fact_inpatient = fact_inpatient.withColumn(

    "LENGTH_OF_STAY",

    datediff(
        col("CLM_THRU_DT"),
        col("CLM_FROM_DT")
    )

)

# ==========================================================
# Insurance / Patient Payment
# FIX: CLM_PMT_AMT is what Medicare actually paid — this dataset has
# no separate "billed amount" field for Inpatient claims. The old
# version used NCH_PRMRY_PYR_CLM_PD_AMT (a DIFFERENT primary payer's
# contribution, ~always $0) as insurance payment, which made
# PATIENT_PAYMENT end up as ~= CLM_PMT_AMT — i.e. Medicare's real
# payment was being mislabeled as patient payment.
#
# Correct mapping:
#   INSURANCE_PAYMENT = CLM_PMT_AMT (what Medicare actually paid)
#   PATIENT_PAYMENT   = deductible + Part A coinsurance liability
#                        (the fields that actually represent what the
#                        patient owes)
# ==========================================================

fact_inpatient = fact_inpatient.withColumn(
    "INSURANCE_PAYMENT",
    coalesce(col("CLM_PMT_AMT").cast("double"), lit(0.0))
)

fact_inpatient = fact_inpatient.withColumn(
    "PATIENT_PAYMENT",
    coalesce(col("NCH_BENE_IP_DDCTBL_AMT").cast("double"), lit(0.0))
    + coalesce(col("NCH_BENE_PTA_COINSRNC_LBLTY_AM").cast("double"), lit(0.0))
)

fact_inpatient = fact_inpatient.withColumn(
    "CLAIM_AMOUNT",
    col("INSURANCE_PAYMENT") + col("PATIENT_PAYMENT")
)

# ==========================================================
# Select Columns
# ==========================================================

fact_inpatient = fact_inpatient.select(

    col("CLM_ID"),

    col("BENEFICIARY_KEY"),

    col("PROVIDER_KEY"),

    col("DIAGNOSIS_KEY"),

    col("ADMISSION_DATE_KEY"),

    col("DISCHARGE_DATE_KEY"),

    col("LENGTH_OF_STAY"),

    col("CLAIM_AMOUNT"),

    col("INSURANCE_PAYMENT"),

    col("PATIENT_PAYMENT"),

    col("NCH_BENE_IP_DDCTBL_AMT").alias("DEDUCTIBLE"),

    col("CLM_PASS_THRU_PER_DIEM_AMT").alias("PER_DIEM_PAYMENT"),

    current_timestamp().alias("GOLD_CREATED_TIMESTAMP")

)

# ==========================================================
# Write
# ==========================================================

fact_inpatient.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable(
"healthcare_claims_catalog.gold.fact_inpatient"
)

# ==========================================================
# Validation
# ==========================================================

print("="*60)
print("FACT INPATIENT CREATED")
print("="*60)

total = fact_inpatient.count()
print("Records :", total)
print("Expect this to match the real source claim count (66,514) —")
print("if it doesn't, dim_beneficiary or another joined dimension")
print("still has duplicate keys.")

print("\nInsurance vs Patient payment totals:")
fact_inpatient.select(
    sum("INSURANCE_PAYMENT").alias("total_insurance"),
    sum("PATIENT_PAYMENT").alias("total_patient")
).show(truncate=False)

fact_inpatient.show(10,False)

FACT INPATIENT CREATED
Records : 66165
Expect this to match the real source claim count (66,514) —
if it doesn't, dim_beneficiary or another joined dimension
still has duplicate keys.

Insurance vs Patient payment totals:
+---------------+-------------+
|total_insurance|total_patient|
+---------------+-------------+
|6.293433E8     |7.3298012E7  |
+---------------+-------------+

+---------------+---------------+------------+-------------+------------------+------------------+--------------+------------+-----------------+---------------+----------+----------------+--------------------------+
|CLM_ID         |BENEFICIARY_KEY|PROVIDER_KEY|DIAGNOSIS_KEY|ADMISSION_DATE_KEY|DISCHARGE_DATE_KEY|LENGTH_OF_STAY|CLAIM_AMOUNT|INSURANCE_PAYMENT|PATIENT_PAYMENT|DEDUCTIBLE|PER_DIEM_PAYMENT|GOLD_CREATED_TIMESTAMP    |
+---------------+---------------+------------+-------------+------------------+------------------+--------------+------------+-----------------+---------------+----------+--------------